# SAVI data preparation: example

Runs the data preparation of `SAVI_data_prep.R`, from two Seurat objects to the input file of SAVI (`SAVI_input.rds`). R kernel.

GLEAM is **not** called from here: it runs separately in its own conda environment (`GLEAM_impute.py`, or `GLEAM_example.ipynb` with a Python kernel from that environment). The two sides only exchange files in `work_dir`.

| Part | Step | What | Where |
|---|---|---|---|
| A | 1 | Build / load the two Seurat objects (spatial `P22mouse`, multiome `e18_brain`) | R |
| A | 2 | `SAVI_prepare_gleam_inputs()`: select genes and peaks, write the GLEAM input files | R |
| | 3 | Train GLEAM, write the imputation files | **Python, separate conda environment** |
| B | 4 | `SAVI_load_imputation()`: read the imputation, save `SAVI_input.rds` | R |

Files needed in the working directory: `SAVI_data_prep.R` (self-contained, does not need `SAVI.R`).
The next notebook, `SAVI_example.ipynb`, starts from `SAVI_input.rds`.

## 0. Settings and libraries (edit the paths)

In [8]:
# ---- paths: edit these ----
work_dir      <- "/storage10/shuyang/MISAR-seq-main/More_evaluated_gene"      # all GLEAM files and SAVI_input.rds go here
p22_file      <- "/storage10/shuyang/MISAR-seq-main/E15_5-S1_SpatialSeurat_raw.Rdata"   # contains the object P22mouse
e18_h5        <- "/storage10/shuyang/GLEAM_real/P22_E18/e18_mouse_brain_fresh_5k_filtered_feature_bc_matrix.h5"
e18_fragments <- "/storage10/shuyang/GLEAM_real/P22_E18/e18_mouse_brain_fresh_5k_atac_fragments.tsv.gz"
p22_fragments <- "Data/E15_5-S1_filtered_fragments.tsv.bgz"

dir.create(work_dir, recursive = TRUE, showWarnings = FALSE)

library(Seurat)
library(Signac)
library(GenomicRanges)
library(Matrix)
library(data.table)
library(EnsDb.Mmusculus.v79)

source("SAVI_data_prep.R")   # SAVI_prepare_gleam_inputs(), SAVI_load_imputation()

## 1. The two Seurat objects

* `P22mouse`: spatial object with assays `Spatial` (RNA counts) and `peaks`, reduction `spatial`, metadata `RNA_clusters`.
* `e18_brain`: single-cell multiome reference. It is built once with `process_multiome()` (defined below) and cached in `work_dir`.

In [2]:
# process_multiome(): builds the multiome reference object (RNA + ATAC + WNN)
process_multiome <- function(h5_path, fragments_path,annotation, genome = "mm10", resolution = 0.5, pca_dims = 30, lsi_dims = 30){

  # 1. Read data from HDF5
  counts <- Read10X_h5(filename = h5_path)

  # Extract RNA and ATAC counts
  rna_counts <- counts$`Gene Expression`
  atac_counts <- counts$Peaks
  shared_cells = intersect(colnames(rna_counts),colnames(atac_counts))
  # 2. Create RNA Seurat Object
  multiome <- CreateSeuratObject(counts = rna_counts[,shared_cells], assay = "RNA",min.cells = 10)

  # 3. Create ATAC Chromatin Assay

  chrom_assay <- CreateChromatinAssay(
    counts = atac_counts[,shared_cells],
    sep = c(":", "-"),  # Adjust if needed
    genome = genome,
    fragments = fragments_path,
      annotation=annotation
  )


  # 4. Merge RNA and ATAC Objects
  multiome[['ATAC']] = chrom_assay

  # 5. Preprocess RNA
  multiome <- FindVariableFeatures(multiome,assay='RNA',n.features=1000)
    multiome<-NormalizeData(multiome,assay='RNA')
    multiome <- ScaleData(multiome,assay='RNA')
  multiome <- SCTransform(multiome, assay = "RNA", verbose = FALSE)
  multiome <- RunPCA(multiome, assay = "RNA", verbose = FALSE)

  # 6. Preprocess ATAC
    DefaultAssay(multiome) <- 'ATAC'
  multiome <- RunTFIDF(multiome)
  multiome <- FindTopFeatures(multiome, min.cutoff = 10)
  multiome <- RunSVD(multiome, reduction.name = "lsi", reduction.key = "LSI_")

  # 7. Integrate RNA and ATAC Using WNN
  multiome <- FindMultiModalNeighbors(
    multiome,
    reduction.list = list("pca", "lsi"),
    dims.list = list(1:pca_dims, 2:lsi_dims)
  )

  multiome <- RunUMAP(multiome,nn.name = "weighted.nn",reduction.name = "wnn.umap",reduction.key = "wnnUMAP_")
  multiome <- FindClusters(multiome,graph.name = "wsnn", resolution = resolution)

  # 9. Return Processed Seurat Object
  return(multiome)
}

In [3]:
# spatial object
load(p22_file)                                    # -> P22mouse

# multiome reference (built once, then read from the cache)
e18_rds <- file.path(work_dir, "e18_brain.rds")
if (file.exists(e18_rds)) {
  e18_brain <- readRDS(e18_rds)
} else {
  annotations <- GetGRangesFromEnsDb(ensdb = EnsDb.Mmusculus.v79)
  seqlevelsStyle(annotations) <- "UCSC"
  genome(annotations) <- "mm10"
  e18_brain <- process_multiome(h5_path = e18_h5,
                                fragments_path = e18_fragments,
                                annotation = annotations)
  saveRDS(e18_brain, e18_rds)
}

P22mouse
e18_brain

An object of class Seurat 
173705 features across 1949 samples within 2 assays 
Active assay: Spatial (32285 features, 0 variable features)
 1 layer present: counts
 1 other assay present: peaks
 1 dimensional reduction calculated: spatial

An object of class Seurat 
181145 features across 4881 samples within 3 assays 
Active assay: ATAC (144437 features, 138938 variable features)
 2 layers present: counts, data
 2 other assays present: RNA, SCT
 3 dimensional reductions calculated: pca, lsi, wnn.umap

## 2. Prepare the GLEAM inputs

Selects genes and peaks and writes everything GLEAM reads into `work_dir`. Defaults ("Setting 1"): 60,000 sc variable peaks, 1,000 sc variable genes, 1,000 spatial variable peaks, genes with more than 50 peaks in a +/-500 kb window.
Other options: `n_sc_var_peaks`, `n_sc_var_genes`, `n_sp_var_peaks`, `window`, `min_peaks_per_gene`, `additional_gene`, `k`, `sp_reduction`, `cell_type_col`.

In [23]:
# ---- genes from the gold-standard gene-CRE table: top 200 most variable ----
gene_cre_file <- '/storage10/shuyang/GLEAM_real/AdultMB_multiplex/gene-cCREcorrelations.txt'   # <- your gold standard; needs a column `Gene`
n_gold_genes  <- 2500
sc_rna_assay  <- "RNA"                                  # same assays as in SAVI_prepare_gleam_inputs()
sp_rna_assay  <- "Spatial"

gold_genes <- unique(read.table(gene_cre_file, header = TRUE)$Gene)

# only genes measured in BOTH the single-cell and the spatial data can be analysed
gold_genes <- Reduce(intersect, list(gold_genes,
                                     rownames(e18_brain[[sc_rna_assay]]),
                                     rownames(P22mouse[[sp_rna_assay]])))

# variability of every gene in the single-cell data (same measure FindVariableFeatures() uses)
tmp <- FindVariableFeatures(e18_brain, assay = sc_rna_assay, selection.method = "vst",
                            nfeatures = 6000, verbose = FALSE)
hv   <- Seurat::HVFInfo(tmp, assay = sc_rna_assay)
vcol <- grep("variance.standardized", colnames(hv), value = TRUE)[1]
rm(tmp)

# top 200 most variable among the gold-standard genes
hv_gold <- hv[gold_genes, , drop = FALSE]
hv_gold <- hv_gold[order(hv_gold[[vcol]], decreasing = TRUE), , drop = FALSE]
gold_top <- head(rownames(hv_gold)[!is.na(hv_gold[[vcol]])], n_gold_genes)

cat("gold-standard genes in both datasets:", length(gold_genes),
    "| selected:", length(gold_top), "\n")

gold-standard genes in both datasets: 11973 | selected: 2500 


In [24]:
sel <- SAVI_prepare_gleam_inputs(
  sc_data = e18_brain,
  sp_data = P22mouse,
  fragment_path = p22_fragments,
  work_dir = work_dir,
  n_sc_var_genes = 100,
  n_sp_var_peaks = 60000, additional_gene = gold_top
)

length(sel$use_genes)     # genes SAVI will analyse
length(sel$sc_peaks)      # sc peaks near those genes
length(sel$sp_peaks)      # spatial-specific peaks
list.files(work_dir)      # csv files read by GLEAM

Step 1: selecting genes and peaks

  genes: 356 | sc peaks: 12282 | spatial-specific peaks: 55488

Step 2: building raw data matrices

Step 3: joint embedding and kNN graphs

Using RSpectra for CCA since the cell-by-cell cross-product matrix has at least 1,000,000 entries.
To use irlba, set `svd.method = "irlba"`
`vst.flavor` is set to 'v2' but could not find glmGamPoi installed.
Please install the glmGamPoi package for much faster estimation.
--------------------------------------------
install.packages('BiocManager')
BiocManager::install('glmGamPoi')
--------------------------------------------
Falling back to native (slower) implementation.


Warning message in theta.ml(Y, mu, sum(w), w, limit = control$maxit, trace = control$trace > :
"iteration limit reached"
Warning message in sqrt(1/i):
"NaNs produced"
Warning message in theta.ml(Y, mu, sum(w), w, limit = control$maxit, trace = control$trace > :
"iteration limit reached"
Warning message in sqrt(1/i):
"NaNs produced"
Warning mess

[1] 356

[1] 12282

[1] 55488

[1] "Data"                             "E15_5-S1_SpatialSeurat_raw.Rdata"
 [3] "E15_5-S1_SpatialSeurat_raw.rds"   "E15_pseudoT.csv"                 
 [5] "e18_brain.rds"                    "generated_data.Rdata"            
 [7] "SAVI_baselines.rds"               "SAVI_evaluation.rds"             
 [9] "SAVI_input.rds"                   "SAVI_result_sp5_t3.Rdata"        
[11] "SAVI_selection.rds"               "sc_atac_scaled.csv"              
[13] "sc_cca_emb.csv"                   "sc_cca_knn.csv"                  
[15] "sc_rna_atac_knn.csv"              "sc_rna_raw.csv"                  
[17] "sp_atac_exist_withatac.csv"       "sp_atac_scaled.csv"              
[19] "sp_cca_emb.csv"                   "sp_cca_knn.csv"                  
[21] "sp_rna_atac_knn.csv"              "sp_RNA_clusters.csv"             
[23] "sp_rna_exist_withatac_all.csv"    "sp_rna_pred_withatac_all.csv"    
[25] "sp_rna_raw.csv"                   "sp_spatial_coord.csv"            
[27] "sp_spatial_knn.csv"

In [25]:
sum(is.element(gold_top, sel$use_genes))

[1] 355

## 3. Run GLEAM separately (in its own conda environment)

Part A is finished. Everything GLEAM needs is now in `work_dir`. Run GLEAM in the conda environment that has its requirements (torch, torch_geometric, ...), either

* in a terminal:

  ```bash
  conda activate <gleam_env>
  python GLEAM_impute.py --work_dir /storage10/shuyang/GLEAM_real/P22_E18 --gpu 1
  ```

* or with `GLEAM_example.ipynb` (Python kernel of that environment).

When it has finished, `work_dir` contains `sp_rna_exist_withatac_all.csv`, `sp_rna_pred_withatac_all.csv` and `sp_atac_exist_withatac.csv`. Then continue with Part B below. If you closed R in the meantime, just run the settings cell and the `source()` line again (or the small setup cell in Part B).

---
# Part B: after GLEAM has finished

In [ ]:
# Part B setup (only needed if you restarted R after Part A)
work_dir <- "/storage10/shuyang/MISAR-seq-main"
source("SAVI_data_prep.R")

# the three files written by GLEAM; stop here if they are not there yet
gleam_out <- c("sp_rna_exist_withatac_all.csv", "sp_rna_pred_withatac_all.csv", "sp_atac_exist_withatac.csv")
found <- setNames(file.exists(file.path(work_dir, gleam_out)), gleam_out)
print(found)
if (!all(found)) stop("GLEAM output not found in work_dir: run GLEAM first (step 3).")

## 4. Load the imputation and save the SAVI input

`SAVI_load_imputation()` returns the observed data (`raw`), the imputed data (`imputed`, cells x features) and the other SAVI inputs.
`restrict_genes` (optional) keeps only genes that have a MultiVelo pseudotime estimate; it can also be applied later in the SAVI notebook.

In [ ]:
prep <- SAVI_load_imputation(work_dir)

length(prep$use_genes)
dim(prep$imputed$rna)       # spatial cells x genes
dim(prep$imputed$atac)      # spatial cells x peaks
dim(prep$raw$rna); dim(prep$raw$atac)
head(prep$spatial_coord, 3)
table(prep$cell_types)
head(prep$use_peaks[[1]])   # candidate peaks of the first gene

In [ ]:
saveRDS(prep, file.path(work_dir, "SAVI_input.rds"))
file.exists(file.path(work_dir, "SAVI_input.rds"))